In [4]:
import math

In [5]:
def class_of_bolt(grade:float)->tuple[float,float]:
    """
    Returns fub and fyb
    """
    fub=int(grade)*100
    dec=grade-int(grade)
    fyb=dec*fub
    return fub,round(fyb,1)

In [2]:
class_of_bolt(4.6)

(400, 240.0)

In [6]:
def net_area_of_bolt(d:float)->tuple[float,float]:
    """
    Returns the net area of the bolt,Anb(mm2)
    where d- diameter of the bolt,mm 
    """
    Asb=math.pi/4*(d**2)
    Anb=0.78*Asb

    return round(Anb,2),round(Asb,2)

In [25]:
net_area_of_bolt(20)

(245.04, 314.16)

In [7]:
def hole_diameter(d:float)->float:
    """
    IS 800:2007 (Clause 10.2.1), the diameter of a standard bolt hole (d₀) is larger than the nominal diameter of the bolt (d) 
    to allow easy insertion and account for minor misalignments. For standard clearance holes, the extra clearance added to the bolt 
    diameter depends on the bolt size: 1.0 mm extra for 12 to 14 mm bolts, 2.0 mm extra for 16 to 24 mm bolts, and 3.0 mm extra for bolts 
    larger than 24 mm.
    """
    if d>12 and d<14:
        do=d+1
    elif d>16 and d<24:
        do=d+2
    elif d>24:
        do=d+3

    return do

In [10]:
hole_diameter(20)

22

In [8]:
def edge_distance (d:float,t:float,fy:float)->tuple[float,float]:
    """
    Returns the maximum and minimum edge/end distance
    """
    do=hole_diameter(d)
    emin=1.5*do
    eps=(250/fy)**(1/2)
    emax=12*t*eps

    return emin,emax

In [12]:
edge_distance(20,10,250)

(33.0, 120.0)

In [9]:
def pitch(d:float,t:float,type:str)->tuple[float,float]:
    """
    Returns the minimum pitch,p_min,mm & maximum pitch,p_max,mm
    d-diameter of the bolt,mm
    t-thickness of the thinner plate,mm
    type-type of member(tension/compression)

    IS 800:2007 (Clause 10.2.2), The distance between centre of fasteners shall not be
    less than 2.5 times the nominal diameter of the fastener.
    """
    p_min=2.5*d
    if (type=='compression'):
        p_max=min(12*t,200)
    elif (type=='tension'):
        p_max=min(16*t,200)
    return p_min,p_max



In [14]:
pitch(20,10,'compression')[0]

50.0

In [10]:
def shear_strength_of_bolt(grade:float,d:float,nn:int,ns:int,lj:float,lg:float,tpk:float)->float:
    """
    returns the shear strength of the bolt
    """
    fub=class_of_bolt(grade)[0]
    Anb,Asb=net_area_of_bolt(d)
    if lj>(15*d):
     rf_lj=1.075-(lj/(200*d))
     rf_lj = max(0.75, min(rf_lj, 1.0))
    else:
     rf_lj=1
    if lg>(5*d):
     rf_lg=(8*d)/(lg+(3*d))
     rf_lg = min(rf_lg, rf_lj)
    else:
     rf_lg=1
    if tpk>6:
     rf_pk=1-(0.0125*tpk)
    else:
     rf_pk=1
    V_dsb=(fub/(math.sqrt(3)*1.25))*((nn*Anb)+(ns*Asb))*rf_lg*rf_lj*rf_pk*10**-3
    return round(V_dsb,2)

In [29]:
shear_strength_of_bolt(4.6,20,1,0,150,50,5)

45.27

In [11]:
def tensile_strength_of_bolt(grade:float,d:float)->float:
    """
    returns the tensile strength of the bolt
    """
    fub=class_of_bolt(grade)[0]
    Anb=net_area_of_bolt(d)[0]
    T_db=(0.9*fub*Anb)/1.25*10**-3
    return round(T_db,2)
    

In [7]:
tensile_strength_of_bolt(4.6,20)

70.57

In [12]:
def bearing_strength_of_bolt(d:float,fu:float,grade:float,tmin:float,t:float)->float:
    """
    returns the bearing strength of the bolt
    """
    e=edge_distance(d,tmin,fu)[0]
    p=pitch(d,tmin,'compression')[0]
    do=hole_diameter(d)
    fub=class_of_bolt(grade)[0]
    kb=min((e/(3*do)),((p/(3*do))-0.25),(fu/fub),1)
    V_dpb=(2.5*kb*fu*(d*t))/1.25*10**-3
    return round(V_dpb,2)

In [10]:
bearing_strength_of_bolt(20,410,4.6,14,28)

229.6

In [13]:
def design_strength_of_bolt(grade:float,d:float,nn:int,ns:int,lj:float,lg:float,tpk:float,
                            fu:float,tmin:float,t:float)->float:
    """
    """
    V_dsb=shear_strength_of_bolt(grade,d,nn,ns,lj,lg,tpk)
    V_dpb=bearing_strength_of_bolt(d,fu,grade,tmin,t)
    T_db=tensile_strength_of_bolt(grade,d)
    Vb=min(V_dsb,V_dpb,T_db)
    return round(Vb,2)
    
    
    
    

In [14]:
design_strength_of_bolt(4.6,20,1,0,150,50,5,410,14,28)

45.27

In [1]:
def design_strength_of_butt_weld(lw:float,tmin:float,fy:float,fu:float,weld_type:str,weld_penetration:str)->tuple[float,float]:
   """
   returns the design strength of weld
   """
   if (weld_penetration=='single'):
    te=(5/8)*tmin
    f=fu
   elif (weld_penetration=='double'):
    te=tmin
    f=fy
   if (weld_type=='shop weld'):
      psf=1.25
   elif (weld_type=='field weld'):
      psf=1.5

   T_dw=(f/psf)*(lw*te)*10**-3
   V_dw=0.57*(f/psf)*(lw*te)*10**-3

   return round(T_dw,2),round(V_dw,2)

In [3]:
design_strength_of_butt_weld(150,12,240,250,'shop weld','single')[0]

225.0

In [17]:
def angle_of_fusion(theta:float)->float:
    """
    """
    if 60<theta<90:
        K=0.7
    elif 91<theta<100:
        K=0.65
    elif 101<theta<106:
        K=0.6
    elif 107<theta<113:
        K=0.55
    elif 114<theta<120:
        K=0.5
    else:
        K=0.7
    return K
        

In [18]:
angle_of_fusion(60)

0.7

In [19]:
def design_strength_of_fillet_weld(s:float,lw:float,weld_type:str,fu:float,theta:float)->tuple[float,float]:
    """
    returns the design strength of weld
    """
    K=angle_of_fusion(theta)
    tt=K*s
    if (weld_type=='shop weld'):
          psf=1.25
    elif (weld_type=='field weld'):
          psf=1.5
    T_dw=(fu/psf)*(lw*tt)*10**-3
    V_dw=(fu/(math.sqrt(3)*psf))*(lw*tt)*10**-3
    return round(T_dw,2),round(V_dw,2)
    

In [20]:
design_strength_of_fillet_weld(6,216,'shop weld',410,60)

(297.56, 171.8)

In [2]:
import pandas as pd


In [5]:
df=pd.read_csv('steel_tables_is.csv')

In [6]:
df=df.set_index('Section')

In [7]:
rolled_steel_beam=df.copy()

In [8]:
rolled_steel_beam

,W_kg/m,W_N/m,Area,h,bf,tf,tw,Ixx,Iyy,rxx,...,r1,r2,D_deg,h1,h2,b1,C,g,g1_min,Max_flange_rivet_mm
Section,,,,,,,,,,,,,,,,,,,,,
ISJB 150,7.1,69.7,9.01,150,50,4.6,3.0,322.1,9.2,5.98,...,5.0,1.5,91.5,130.4,9.80,23.50,3.00,30,45.0,6
ISJB 175,8.1,79.5,10.28,175,50,4.8,3.2,479.3,9.7,6.83,...,5.0,1.5,91.5,155.0,10.00,23.40,3.10,30,45.0,6
ISJB 200,9.9,97.1,12.64,200,60,5.0,3.4,780.7,17.3,7.86,...,5.0,1.5,91.5,179.5,10.25,28.38,3.20,30,45.0,6
ISJB 225,12.8,125.6,16.28,225,80,5.0,3.7,1308.5,40.5,8.97,...,6.5,1.5,91.5,201.5,11.95,38.15,3.35,40,45.0,12
ISLB 75,6.1,59.8,7.71,75,50,5.0,3.7,72.7,10.0,3.07,...,6.5,2.0,91.5,51.7,11.65,23.15,3.35,30,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ISHB 350,72.4,710.2,92.21,350,250,11.6,10.1,19802.8,2510.5,14.65,...,12.0,6.0,94.0,296.0,27.00,119.95,6.55,140,60.0,32
ISHB 400,77.4,759.3,98.66,400,250,12.7,9.1,28063.5,2728.3,16.87,...,14.0,7.0,94.0,340.1,29.90,120.45,6.05,140,65.0,32
ISHB 400,82.2,806.4,104.68,400,250,12.7,10.6,28823.5,2783.0,16.61,...,14.0,7.0,94.0,340.1,29.90,119.70,6.80,140,65.0,32


In [3]:
def rolled_steel_beam(beam:str,W:float)->tuple[float,float,float,float,float,float,str]:
    """
    """
    df=pd.read_csv('steel_tables_is.csv')
    df=df.set_index('Section')
    rolled_steel_beam=df.copy()
    condition = ((rolled_steel_beam.index == beam) & (rolled_steel_beam["W_N/m"] == W))
    steel_beam = rolled_steel_beam.loc[condition].iloc[0]
    

    Area = steel_beam["Area"]
    h = steel_beam["h"]
    bf = steel_beam["bf"]
    tf = steel_beam["tf"]
    rzz = steel_beam["rzz"]
    ryy = steel_beam["ryy"]

    rmin=min(rzz,ryy)
    if h/bf>1.2 and tf<=40:
        if rmin==rzz:
            buckling_class='a'
        elif rmin==ryy:
            buckling_class='b'
    elif h/bf>1.2 and 40<=tf<=100:
        if rmin==rzz:
            buckling_class='b'
        elif rmin==ryy:
            buckling_class='c'
    elif h/bf<=1.2 and tf<=100:
        if rmin==rzz:
            buckling_class='b'
        elif rmin==ryy:
            buckling_class='c'
    elif h/bf<=1.2 and tf>100:
        if rmin==rzz:
            buckling_class='d'
        elif rmin==ryy:
            buckling_class='d'

    return Area, h, bf, tf, rzz, ryy,buckling_class
    


In [5]:
rolled_steel_beam('ISHB 350',710.20)

(92.21, 350, 250, 11.6, 14.65, 5.22, 'b')

In [11]:
def effective_length_factor(condition: str) -> float:
    """
    Returns effective length factor K
    as per IS 800:2007 Table 11.
    """

    if condition == "fixed_fixed":
        K = 0.65

    elif condition == "fixed_pinned":
        K = 0.80

    elif condition == "pinned_pinned":
        K = 1.00

    elif condition == "fixed_guided":
        K = 1.20

    elif condition == "fixed_free":
        K = 2.00

    else:
        raise ValueError("Invalid end restraint condition")

    return K

In [13]:
effective_length_factor('fixed_fixed')

0.65

In [15]:
def design_compressive_strength(beam:str,W:float,L:float,fy:float,psf:float,condition:str)->float:
    """
    to determine the design compressive strength of the member
    """
    section=rolled_steel_beam(beam,W)
    Area=section[0]
    rzz,ryy=section[4],section[5]
    buckling_class=section[6]
    rmin=min(rzz,ryy)*10
    K=effective_length_factor(condition)
    E=2*10**5
    
    fcc=(math.pi**2*E)/(((K*L*1000)/rmin)**2)
    lamda=math.sqrt(fy/fcc)
    buckling_class=section[6]
    alpha=imperfection_factor(buckling_class)
    phi=0.5*(1+(alpha*(lamda-0.2))+(lamda**2))
    fcd=(fy/psf)/(phi+math.sqrt((phi**2)-(lamda**2)))
    Pcd=fcd*Area*100*10**-3

    return round(Pcd,2)
    
    
    
    

In [16]:
design_compressive_strength('ISHB 350',710.20,4,250,1.1,'fixed_fixed')

1794.69

In [7]:
def imperfection_factor(buckling_class:str)->float:
    """
    """
    if (buckling_class=='a'):
        alpha=0.21
    elif (buckling_class=='b'):
        alpha=0.34
    elif (buckling_class=='c'):
        alpha=0.49
    elif (buckling_class=='d'):
        alpha=0.76
        
    return alpha
    

In [87]:
imperfection_factor('b')

0.34

In [22]:
def gross_section_yielding(fy:float,Ag:float)->float:
    """
    """
    gamma_m0=1.1
    T_dg=(fy/gamma_m0)*Ag*10**-3
    return round(T_dg,2)
    

In [5]:
gross_section_yielding(250,1200)

272.73

In [23]:
def net_section_rupture_plates(fu:float,Anet:float)->float:
    """
    """
    gamma_m1=1.25
    T_dn=(0.9*fu/gamma_m1)*Anet*10**-3
    return round(T_dn,2)
    

In [7]:
net_section_rupture_plates(410,760)

224.35

In [24]:
def net_section_rupture_angles_channels(w:float,t:float,fu:float,fy:float,Ago:float,Anc:float,bs:float,Lc:float)->float:
    """
    """
    gamma_m0=1.1
    gamma_m1=1.25
    beta=1.4-(0.076*(w/t)*(fy/fu)*(bs/Lc))
    T_dn=(((beta*fy/gamma_m0)*Ago)+((0.9*fu/gamma_m1)*Anc))*10**-3
    return round(T_dn,2)

In [14]:
net_section_rupture_angles_channels(60,8,410,250,448,512,100,120)

264.2

In [25]:
def block_shear_failure(fu:float,fy:float,Avg:float,Atn:float,Avn:float,Atg:float)->float:
    """
    """
    gamma_m0=1.1
    gamma_m1=1.25
    T_db1 = ((fy * Avg) / (math.sqrt(3) * gamma_m0)+ (0.9 * fu * Atn) / gamma_m1) * 10**-3
    T_db2 = ((0.9 * fu * Avn) / (math.sqrt(3) * gamma_m1)+ (fy * Atg) / gamma_m0) * 10**-3
    T_db= min(T_db1,T_db2)
    return round(T_db,2)

In [18]:
block_shear_failure(410,250,1600,190,1050,300)

247.14

In [29]:
def design_tensile_strength(fy:float,Ag:float,fu:float,Anet:float,Avg:float,Atn:float,Avn:float,Atg:float)->float:
    """
    Returns the design tensile strength of the member
    """
    gamma_m0=1.1
    gamma_m1=1.25
    T_dg=gross_section_yielding(fy,Ag)
    T_dn=net_section_rupture_plates(fu,Anet)
    T_db=block_shear_failure(fu,fy,Avg,Atn,Avn,Atg)
    T_d=min(T_dg,T_dn,T_db)
    return round(T_d,2)
    
    

In [30]:
design_tensile_strength(250,1500,410,1060,1800,390,1300,500)

312.91